In [69]:
import bs4
import requests as r
from xml.etree import ElementTree as ET
import pandas as pd
from pandas import DataFrame
import sqlite3 as sql
META_COURSES_URL = 'https://courses.rice.edu/courses/!SWKSCAT.info'
BASE_COURSES_URL = 'https://courses.rice.edu/'
BASE_GA_URL = 'https://ga.rice.edu'

def get_term_codes() -> DataFrame:    
    req = r.get(f"{META_COURSES_URL}?action=TERMS", timeout=15) 
    terms = ET.fromstring(req.text).findall('TERM')
    df = []

    for term in terms:
        # ignore quadmesters since they are only for the Glasscock school, not undergraduate
        if not "Quadmester" in term.find('OPT').tail:
            df.append({
                'term': term.find('OPT').tail,
                'code': term.attrib.get('code')
            })

    return DataFrame(df)

def get_subject_codes(term_code: str, export_to_sql=False) -> DataFrame:
    req = r.get(f"{META_COURSES_URL}?action=SUBJECTS&term={term_code}", timeout=15) 
    subjects = ET.fromstring(req.text).findall('SUBJECT')
    df = []

    for subject in subjects:
        df.append({
            'subject': subject.find('OPT').tail,
            'code': subject.attrib.get('code')
        })
    
    df = DataFrame(df)

    if export_to_sql:
        con = sql.connect(f'subjects_{term_code}.db')
        try:
            df.to_sql(f'subjects_{term_code}.db', con)
        except ValueError as e:
            print(e)

    return df

def get_course_syllabus(term_code: str, crn: str):
    req = r.get(f"{META_COURSES_URL}?action=SYLLABUS&term={term_code}&crn={crn}", timeout=15)
    res = ET.fromstring(req.text)
    if res.attrib.get('has-syllabus') != 'yes':
        return None
    return res.attrib.get('doc-url')

def get_schools(term_code: str, export_to_sql=False) -> DataFrame:
    req = r.get(f"{META_COURSES_URL}?action=SCHOOLS&term={term_code}", timeout=15) 
    schools = ET.fromstring(req.text).findall('SCHOOL')
    df = []

    for school in schools:
        df.append({
            'school': school.find('OPT').tail,
            'code': school.attrib.get('code')
        })

    df = DataFrame(df)
    if export_to_sql:
        con = sql.connect(f'schools_{term_code}.db')
        try:
            df.to_sql(f'schools_{term_code}.db', con)
        except ValueError as e:
            print(e)

    return df

In [70]:
def get_course_info(term_code: str, school_code: str) -> DataFrame:
    courses = []
    req = r.get(f"{BASE_COURSES_URL}/courses/courses/!SWKSCAT.cat?p_action=QUERY&p_term={term_code}&p_school={school_code}", timeout=15)
    parser = bs4.BeautifulSoup(req.text, 'html.parser')
    rows = parser.find('tbody').find_all('tr')
    rows[0].find_all('td')

    for row in parser.find('tbody').find_all('tr'):
        cells = row.find_all('td')
        if len(cells) != 7:
            raise ValueError(f"Expected 7 cells per row, got {len(cells)}")
        courses.append({
            'crn': cells[0].text,
            'crs': cells[1].text,
            'title': cells[3].text, #ignore "part of term" entry
            'instructors': cells[4].text,
            'meeting_times': cells[5].find('div').text.strip(), # ignore final exam time
            'credits': cells[6].text,
            'course_page': f"{BASE_COURSES_URL}{cells[0].a['href']}"
        })

    return DataFrame(courses)

def get_course_description(term_code: str, crn: str) -> str:
    req = r.get(f"{BASE_COURSES_URL}/courses/courses/!SWKSCAT.cat?p_action=COURSE&p_term={term_code}&p_crn={crn}", timeout=15)
    if req.status_code != 200:
        raise ValueError(f"Failed to get course description for {term_code} {crn}")
    parser = bs4.BeautifulSoup(req.text, 'html.parser')
    return parser.find_all('b')[-1].parent.text.split('Description: ')[-1].strip()

def get_all_courses(term_code: str, export_to_csv = False, export_to_sql = False) -> DataFrame:
    schools = get_schools(term_code)
    all_courses = []

    for school_code in schools['code']:
        all_courses.append(get_course_info(term_code, school_code))
    
    result = pd.concat(all_courses, ignore_index=True)
    if export_to_csv:
        result.to_csv(f'courses_{term_code}.csv', index=False)
    if export_to_sql:
        con = sql.connect(f'courses_{term_code}.db')
        try: 
            result.to_sql(f'courses_{term_code}.db', con, index=False)
        except ValueError as e:
            print(e)

    return result

In [71]:
def get_programs() -> DataFrame:
    req = r.get(f"{BASE_GA_URL}/programs-study/", timeout=15)
    parser = bs4.BeautifulSoup(req.text, 'html.parser')
    df = []
    # find ul content with tag "class"="sitemap"
    programs = parser.find('div', class_='sitemap').find_all('li')
    for program in programs:
        df.append({
            'program': program.text,
            'url': program.a['href']})
    return DataFrame(df)

In [ ]:
# fuzzy match courses.db

entry = ""

# read courses.db
con = sql.connect('courses_202620.db')
df = pd.read_sql_query("SELECT * FROM courses_202620", con)

# filter by crs, title and instructors

df_filtered = df[df['crs'].str.contains(entry, case=False) | df['title'].str.contains(entry, case=False) | df['instructors'].str.contains(entry, case=False)]
df_filtered

,crn,crs,title,instructors,meeting_times,credits,course_page
18,20022,ARCH 402 001,ADVANCED TOPICS II,"Nguyen, Tung",1:00PM - 5:00PM MWF 2-MAY-2026 2:00PM - 5:00...,6,https://courses.rice.edu//courses/courses/!SWK...
19,21141,ARCH 402 002,ADVANCED TOPICS II,"Ting, Ian",1:00PM - 5:00PM MWF 2-MAY-2026 2:00PM - 5:00...,6,https://courses.rice.edu//courses/courses/!SWK...
81,20085,BIOE 332 001,BIOENGINEERING THERMODYNAMICS,"Tabor, JeffreyLu, George",10:00AM - 10:50AM MWF 30-APR-2026 2:00PM - 5...,3,https://courses.rice.edu//courses/courses/!SWK...
86,20086,BIOE 372 001,BIOMECHANICS,"Szablowski, JerzyGrande-Allen, K. Jane",9:25AM - 10:40AM TR 1-MAY-2026 9:00AM - 12:0...,3,https://courses.rice.edu//courses/courses/!SWK...
94,20195,BIOE 484 001,BIOPHOTONICS INSTRUMENTATION,"Tkaczyk, Tomasz",1:00PM - 2:15PM TR 3-MAY-2026 2:00PM - 5:00P...,3,https://courses.rice.edu//courses/courses/!SWK...
...,...,...,...,...,...,...,...
4504,22073,SOSC 302 001,SOCIAL SCIENCES STATISTICS,"Ferguson, Todd",10:00AM - 10:50AM MWF 30-APR-2026 2:00PM - 5...,4,https://courses.rice.edu//courses/courses/!SWK...
4505,22074,SOSC 302 002,SOCIAL SCIENCES STATISTICS,"Ferguson, Todd",11:00AM - 11:50AM MWF 1-MAY-2026 2:00PM - 5:...,4,https://courses.rice.edu//courses/courses/!SWK...
4506,25676,SOSC 302 003,SOCIAL SCIENCES STATISTICS,"Li, Jing",9:00AM - 9:50AM MWF 4-MAY-2026 2:00PM - 5:00...,4,https://courses.rice.edu//courses/courses/!SWK...
4558,24346,FWIS 233 001,MATHEMATICS AS METAPHORS,"Yan, Jiajun",1:00PM - 2:15PM TR 3-MAY-2026 2:00PM - 5:00P...,3,https://courses.rice.edu//courses/courses/!SWK...


,crn,crs,title,instructors,meeting_times,credits,course_page
760,25860,ELEC 436 002,FUNDAMENTALS OF CONTROL SYST,"Ghorbel, Fathi",11:00AM - 11:50AM MWF 5-MAY-2026 2:00PM - 5:...,3,https://courses.rice.edu//courses/courses/!SWK...
974,25854,MECH 420 002,FUNDAMENTALS OF CONTROL SYST,"Ghorbel, Fathi",11:00AM - 11:50AM MWF 5-MAY-2026 2:00PM - 5:...,3,https://courses.rice.edu//courses/courses/!SWK...
1028,25857,MECH 620 002,FUNDAMENTALS OF CONTROL SYST,"Ghorbel, Fathi",11:00AM - 11:50AM MWF Scheduled Final Exam-OT...,3,https://courses.rice.edu//courses/courses/!SWK...
2152,25323,MGMT 610 001,FUNDAMENTALS OF THE ENERGY IND,"Spector, Brian",2:15PM - 3:45PM MW GR Course-Dept Schedules Exam,1.5,https://courses.rice.edu//courses/courses/!SWK...
2278,21354,MUSI 117 001,FUNDAMENTALS OF MUSIC I,"Marvin, Tyler",9:00AM - 9:50AM MWF 4-MAY-2026 2:00PM - 5:00...,3,https://courses.rice.edu//courses/courses/!SWK...
2279,24363,MUSI 117 002,FUNDAMENTALS OF MUSIC I,"Sung, Chennie",2:30PM - 3:45PM TR 5-MAY-2026 2:00PM - 5:00P...,3,https://courses.rice.edu//courses/courses/!SWK...
2280,23610,MUSI 117 003,FUNDAMENTALS OF MUSIC I,"Shea, Fiona",11:00AM - 11:50AM MWF 1-MAY-2026 2:00PM - 5:...,3,https://courses.rice.edu//courses/courses/!SWK...
4602,22493,LPAP 103 001,FUNDAMENTALS OF TAI CHI SWORD,"Wu, Chienli",7:00PM - 8:30PM W No Final Exam,1,https://courses.rice.edu//courses/courses/!SWK...
4603,23954,LPAP 103 002,FUNDAMENTALS OF TAI CHI SWORD,"Wu, Chienli",9:25AM - 10:15AM TR No Final Exam,1,https://courses.rice.edu//courses/courses/!SWK...
